# Results Data Cleaning

Olympic athlete competition results data cleaning and preprocessing.

This notebook cleans the raw `results.csv` dataset and prepares it for exploratory data analysis (EDA).

## 1. Import Libraries

In [ ]:
import pandas as pd
import re

## 2. Load Raw Data

The raw Olympic results dataset is loaded into `results_df`.

A copy is created as `results` so that the original raw data remains unchanged during the cleaning process.

In [21]:
results_df = pd.read_csv('./results.csv', low_memory=False)
results = results_df.copy()

## 3. Remove Unnecessary Columns

Columns that are not required for the analysis are removed.

- `Unnamed: 7`: completely empty column.
- `Nationality`: contains very few non-null values and is not required because `NOC` is used for country-level analysis.

In [22]:
results = results.drop(columns=[
    'Unnamed: 7',
    'Nationality'
])

## 4. Extract Year and Season

The Olympic year and season are extracted from the `Games` column.

The `Year` column contains the four-digit year at the beginning of the Games name.

The `Season` column contains `Summer` or `Winter`. Special Games records that do not explicitly specify a season remain missing.

In [23]:
results['Year'] = (
    results['Games']
    .str.extract(r'^(\d{4})', expand=False)
    .astype('Int64')
)

results['Season'] = results['Games'].str.extract(
    r'\b(Summer|Winter)\b',
    expand=False
)

## 5. Clean Position Data

The `pos` column contains different formats, including numeric positions, tied positions, and additional competition information.

Examples include:

- `3.0` → `3`
- `=9` → `9`
- `=17` → `17`
- `3 h1 r2/4` → `3`
- `DNS`, `DNF`, `AC`, `DQ` → non-ranking status

A numeric `pos` value is extracted into the `pos_rank` column, while non-ranking status codes are stored separately in `pos_status`.

In [ ]:
pos_text = results['pos'].astype('string').str.strip()

results['pos_rank'] = (
    pos_text
    .str.extract(r"^=?['\"]?(\d+)", expand=False)
    .astype('Int64')
)

results['pos_status'] = (
    results['pos']
    .astype('string')
    .str.strip()
    .str.extract(r'^([A-Za-z]+)', expand=False)
)

results = results.drop(columns=['pos'])

results = results.rename(columns={
    'pos_rank': 'pos'
})

## 6. Data Correction

One missing `NOC` value was identified for Charlotta Säfvenberg in the 2022 Winter Olympics results.

The athlete's country code was verified externally and corrected to `SWE`.

In [86]:
results.loc[
    results['as'] == 'Charlotta Säfvenberg',
    'noc'
] = 'SWE'

## 7. Standardize Column Names

Column names are converted to lowercase `snake_case` format to provide a consistent naming convention throughout the analysis.

In [32]:
results.columns = (
    results.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s/()-]+', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

## 8. Final Data Check

The cleaned dataset is inspected for its final structure, data types, and missing values before export.

In [111]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 308408 entries, 0 to 308407
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   games       308408 non-null  str   
 1   event       308408 non-null  str   
 2   team        121714 non-null  str   
 3   medal       44139 non-null   str   
 4   as          308408 non-null  str   
 5   athlete_id  308408 non-null  int64 
 6   noc         308408 non-null  str   
 7   discipline  308407 non-null  str   
 8   year        308408 non-null  Int64 
 9   season      305807 non-null  str   
 10  pos         279433 non-null  Int64 
 11  pos_status  27150 non-null   string
dtypes: Int64(2), int64(1), str(8), string(1)
memory usage: 28.8 MB


In [112]:
results.isna().sum()

games              0
event              0
team          186694
medal         264269
as                 0
athlete_id         0
noc                0
discipline         1
year               0
season          2601
pos            28975
pos_status    281258
dtype: int64

## 9. Export Cleaned Data

The cleaned dataset is exported as `results_clean.csv`.

The original `results.csv` file is preserved as the raw dataset.

In [113]:
results.to_csv('results_clean.csv', index=False)